In [0]:
pip install databricks-zerobus-ingest-sdk

In [0]:
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")

In [0]:
import logging
from zerobus.sdk.sync import ZerobusSdk
from zerobus.sdk.shared import RecordType, StreamConfigurationOptions, TableProperties

# See "Get your workspace URL and Zerobus Ingest endpoint" for information on obtaining these values.
SERVER_ENDPOINT="https://<YOUR_SERVER_ENDPOINT>"
DATABRICKS_WORKSPACE_URL="https://<YOUR_WORKSPACE_URL>"
TABLE_NAME=f"{CATALOG}.{SCHEMA}.trading_events"
CLIENT_ID="<YOUR_CLIENT_ID>"
CLIENT_SECRET="<YOUR_CLIENT_SECRET>"

sdk = ZerobusSdk(
    SERVER_ENDPOINT,
    DATABRICKS_WORKSPACE_URL
)

table_properties = TableProperties(TABLE_NAME)
options = StreamConfigurationOptions(record_type=RecordType.JSON)
stream = sdk.create_stream(CLIENT_ID, CLIENT_SECRET, table_properties, options)

try:
    from datetime import datetime, timedelta
    import random
    import uuid

    venues = ["XETRA", "FRA", "STU"]
    order_types = ["LIMIT", "MARKET"]
    sides = ["BUY", "SELL"]
    order_statuses = ["NEW", "PARTIALLY_FILLED", "FILLED", "CANCELLED"]
    instrument_isins = ["DE000BASF111", "DE000AAPL123", "DE000GOOG456"]
    instrument_names = ["BASF", "APPLE", "GOOGLE"]
    participant_ids = ["P1", "P2", "P3"]

    base_time = datetime.utcnow()

    for i in range(1000):
        record_dict = {
            "event_id": str(uuid.uuid4()),
            "timestamp_utc": int((base_time + timedelta(seconds=i)).timestamp() * 1_000_000),
            "instrument_isin": random.choice(instrument_isins),
            "instrument_name": random.choice(instrument_names),
            "venue": random.choice(venues),
            "order_type": random.choice(order_types),
            "side": random.choice(sides),
            "quantity": random.randint(1, 1000),
            "price": str(round(random.uniform(10, 1000), 2)),
            "order_status": random.choice(order_statuses),
            "filled_quantity": random.randint(0, 1000),
            "fill_price": round(random.uniform(10, 1000), 2),
            "participant_id": random.choice(participant_ids),
            "latency_ms": random.randint(1, 100),
            "flag_suspicious": random.choice([True, False]),
            "temp": random.randint(15, 35),
            "humidity": random.randint(30, 90)
        }
        offset = stream.ingest_record_offset(record_dict)

        # Optional: Wait for durability confirmation
        stream.wait_for_offset(offset)
finally:
    stream.close()